# 01 — Chunking + ChromaDB Indexing
**DS593 Final Project**

Reads scraped ToS/Privacy documents, chunks them using LangChain's RecursiveCharacterTextSplitter,
embeds with OpenAI, and stores in ChromaDB for retrieval.


## Step 1: Install Dependencies

In [ ]:
!pip install chromadb openai tiktoken pandas langchain langchain-text-splitters -q
print("✓ Dependencies installed")


## Step 2: Imports + Config

In [ ]:
import os
import json
import re
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Config ────────────────────────────────────────────────────────────────────
CORPUS_DIR  = "tos_corpus"
CHROMA_DIR  = "chroma_db"
MANIFEST    = os.path.join(CORPUS_DIR, "manifest.json")

CHUNK_SIZE    = 2048   # characters
CHUNK_OVERLAP = 256    # characters

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE")
EMBED_MODEL    = "text-embedding-3-small"

# LangChain splitter — tries separators in order:
# paragraph breaks → line breaks → sentences → words → characters
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("✓ Config ready")
print(f"  Chunk size:    {CHUNK_SIZE} chars")
print(f"  Chunk overlap: {CHUNK_OVERLAP} chars")
print(f"  Embed model:   {EMBED_MODEL}")


## Step 3: Load Manifest

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print(f"✓ Loaded {len(manifest)} documents\n")
for doc in manifest:
    print(f"  {doc['company']:15s} | {doc['doc_type']:8s} | {doc['char_count']:,} chars")


## Step 4: Preview Chunking on One Document

Inspect what chunks look like before indexing everything.
Tweak CHUNK_SIZE and CHUNK_OVERLAP in Step 2 if needed.


In [ ]:
# Preview on Spotify ToS
with open(manifest[0]["filepath"], encoding="utf-8") as f:
    text = f.read()

chunks = splitter.split_text(text)

print(f"Document:       {manifest[0]['filepath']}")
print(f"Total chars:    {len(text):,}")
print(f"Total chunks:   {len(chunks)}")
print(f"Avg chunk size: {sum(len(c) for c in chunks) // len(chunks):,} chars")
print()

for i, chunk in enumerate(chunks[:3]):
    print(f"── Chunk {i} ({len(chunk):,} chars) ──")
    print(chunk[:400])
    print("...")
    print()


## Step 5: Metadata Helper

In [ ]:
def extract_metadata_from_filename(filename: str) -> dict:
    """
    Parse company, doc_type, date from filename.
    Format: {company}_{doc_type}_{date}.txt
    e.g. spotify_privacy_2026-04-17.txt
    """
    stem  = Path(filename).stem
    parts = stem.split('_')
    date     = parts[-1]
    doc_type = parts[-2]
    company  = '_'.join(parts[:-2])
    return {"company": company, "doc_type": doc_type, "scrape_date": date}

print("✓ Metadata helper ready")


## Step 6: Set Up ChromaDB

In [ ]:
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBED_MODEL,
)

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if re-running to avoid duplicates
try:
    client.delete_collection("tos_documents")
    print("✓ Deleted existing collection (fresh start)")
except:
    pass

collection = client.get_or_create_collection(
    name="tos_documents",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"},
)

print(f"✓ ChromaDB collection ready: '{collection.name}'")
print(f"  Saved to: {CHROMA_DIR}/")


## Step 7: Index All Documents

Chunks, embeds, and stores every document in ChromaDB.
Batches of 100 chunks per API call to avoid rate limits.

**Cost estimate:** ~$0.05 total for 25 documents.


In [ ]:
total_chunks = 0
skipped      = 0

for doc in manifest:
    filepath = doc["filepath"]
    filename = Path(filepath).name

    if not os.path.exists(filepath):
        print(f"✗ File not found: {filepath}, skipping")
        skipped += 1
        continue

    meta     = extract_metadata_from_filename(filename)
    company  = meta["company"]
    doc_type = meta["doc_type"]
    date     = meta["scrape_date"]

    print(f"\n── {company.upper()} ({doc_type}) ──")

    with open(filepath, encoding="utf-8") as f:
        text = f.read()

    chunks = splitter.split_text(text)
    print(f"  {len(text):,} chars → {len(chunks)} chunks")

    ids       = []
    documents = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        chunk_id = f"{company}_{doc_type}_{date}_{i:04d}"
        ids.append(chunk_id)
        documents.append(chunk)
        metadatas.append({
            "company":     company,
            "doc_type":    doc_type,
            "scrape_date": date,
            "chunk_index": i,
            "source":      filename,
        })

    # Upsert in batches of 100
    batch_size = 100
    for batch_start in range(0, len(ids), batch_size):
        batch_end = batch_start + batch_size
        collection.upsert(
            ids=ids[batch_start:batch_end],
            documents=documents[batch_start:batch_end],
            metadatas=metadatas[batch_start:batch_end],
        )

    print(f"  ✓ Indexed {len(chunks)} chunks")
    total_chunks += len(chunks)

print(f"\n{'='*50}")
print(f"✓ Indexing complete")
print(f"  Total chunks indexed: {total_chunks}")
print(f"  Documents skipped:    {skipped}")
print(f"  Total in collection:  {collection.count()}")


## Step 8: Inspect Collection

In [ ]:
import pandas as pd

all_meta = collection.get()["metadatas"]
df       = pd.DataFrame(all_meta)

summary = df.groupby(["company", "doc_type"]).size().reset_index(name="chunks")
print(f"Collection summary ({collection.count()} total chunks):\n")
print(summary.to_string(index=False))


## Step 9: Sanity Check — Test Retrieval

Run test queries to confirm retrieval is working before building the RAG pipeline.


In [ ]:
def query_collection(question: str, company: str, doc_type: str = None, n_results: int = 3):
    where = {"company": company}
    if doc_type:
        where["doc_type"] = doc_type

    results = collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where,
    )

    print(f"Query:  '{question}'")
    print(f"Filter: company={company}, doc_type={doc_type or 'both'}\n")

    for i, (doc, meta, distance) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    )):
        print(f"── Result {i+1} (distance: {distance:.4f}) ──")
        print(f"   Source: {meta['source']} | chunk {meta['chunk_index']}")
        print(f"   {doc[:300]}...")
        print()

# Test 1
query_collection(
    question="Can Spotify sell my personal data to third parties?",
    company="spotify",
    doc_type="privacy",
)


In [ ]:
# Test 2
query_collection(
    question="Can Discord delete my account without warning?",
    company="discord",
    doc_type="tos",
)


In [ ]:
# Test 3
query_collection(
    question="Does TikTok own the videos I upload?",
    company="tiktok",
)


## Step 10: Build Additional Chunking Collections

Build two more ChromaDB collections using different chunking strategies.
These will be used in 03_evaluation.ipynb to compare retrieval quality.

- `chroma_db_small`    — 512 chars, 64 overlap (fine-grained)
- `chroma_db_sentence` — NLTK sentence boundaries (clause-preserving)

**Cost:** ~2x additional embedding cost (~$0.10 extra total)


In [ ]:
!pip install nltk langchain-text-splitters -q
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from langchain_text_splitters import RecursiveCharacterTextSplitter, NLTKTextSplitter
from pathlib import Path

small_splitter    = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64,
    separators=["\n\n", "\n", ". ", " ", ""]
)
sentence_splitter = NLTKTextSplitter(separator=" ")

EXTRA_STRATEGIES = {
    "chroma_db_small":    small_splitter,
    "chroma_db_sentence": sentence_splitter,
}

print("✓ Additional splitters ready")
print(f"  small:    512 chars, 64 overlap")
print(f"  sentence: NLTK sentence boundaries")


In [ ]:
def build_extra_collection(chroma_dir, splitter, manifest, openai_ef):
    """Index all corpus docs into a new ChromaDB using the given splitter."""
    import chromadb
    client = chromadb.PersistentClient(path=chroma_dir)

    try:
        client.delete_collection("tos_documents")
        print(f"  Deleted existing collection in {chroma_dir}")
    except:
        pass

    col = client.get_or_create_collection(
        name="tos_documents",
        embedding_function=openai_ef,
        metadata={"hnsw:space": "cosine"},
    )

    total = 0
    for doc in manifest:
        filepath = doc["filepath"]
        if not os.path.exists(filepath):
            continue

        filename = Path(filepath).name
        parts    = Path(filename).stem.split("_")
        date     = parts[-1]
        doc_type = parts[-2]
        company  = "_".join(parts[:-2])

        with open(filepath, encoding="utf-8") as f:
            text = f.read()

        chunks    = splitter.split_text(text)
        ids       = [f"{company}_{doc_type}_{date}_{chroma_dir}_{i:04d}" for i in range(len(chunks))]
        metadatas = [{"company": company, "doc_type": doc_type,
                      "scrape_date": date, "chunk_index": i, "source": filename}
                     for i in range(len(chunks))]

        for start in range(0, len(ids), 100):
            col.upsert(
                ids=ids[start:start+100],
                documents=chunks[start:start+100],
                metadatas=metadatas[start:start+100],
            )
        total += len(chunks)

    print(f"  ✓ {chroma_dir}: {total} chunks indexed ({col.count()} in collection)")
    return col


for chroma_dir, splitter in EXTRA_STRATEGIES.items():
    print(f"\nBuilding {chroma_dir}...")
    build_extra_collection(chroma_dir, splitter, manifest, openai_ef)

print("\n✓ All additional collections built")
print("  Ready for chunking comparison in 03_evaluation.ipynb")
